<a href="https://colab.research.google.com/github/TheTrappist/teaching/blob/main/Biochem6761/AU26/04_AU26_classroom_SPR_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First, let's download the data used for this exercise

import pandas as pd # The pandas library will just be used for reading csv files
import numpy as np # import the numpy library that we will need for mathematical operations later

# The following command will parse the file and store it in the variable "data"
data = pd.read_csv('https://raw.githubusercontent.com/TheTrappist/teaching/main/Biochem6761/AU26/sample_data/AU26_02_SPR_IgG_calreticulin.csv')


In [ ]:
# The pandas read_csv command simply takes the data file and saves it to a
# special type of variable called a "dataframe". Think of a dataframe as a
# table, like a table you'd find in Excel or published in a paper. Let's print
# it to see what the whole dataset looks like:

print(data)

You can see our data in a table, with the first column (titled 'time_s') storing the time in seconds and each subsequent column storing the corresponding SPR measurement

In [ ]:
# Now, let's make a simple plot of culture2 optical densities at each time point
from matplotlib import pyplot as plt

plt.style.use("bmh") # This is emphatically NOT required, it's just a visual
# option to make plots look a little nicer. Try removing it to see what happens.

# First, we need to create a figure and an axes object to plot on:

fig, ax = plt.subplots() # the variable "ax" that we got here is the name of the
# new axes object that we can make one or more plots in.

# Let's plot all different functions
time = data['time_s']

for col in data.columns: # Iterate over columns in the data
    if col != 'time_s': # We don't want to plot that stores our x-values
        ax.plot(time, data[col], ls='', marker='o', label=col)

ax.legend()
ax.set_xlabel('Time(s)')
ax.set_ylabel('SPR signal')
plt.show()

Neat! Now let's fit the dissociation and association parts of the dataset in sequence. First, dissociation can be modeled as a single-exponential decay:

$R(t)=R_{0}e^{-k_{off}(t-t_0)}$

Where:

$R(t)$ is the response (signal) at time t,

$R_{0}$ is the response (signal) at the start of the decay,

$k_{off}$ is the off rate (sometimes shown as $k_{-1}$), and

$t_0$ is the time at which the dissociation phase begins

In [ ]:
# First, let's grab only the dissociation phase, which starts around t=600:

data_dissoc = data[data['time_s'] >= 600]

#print(data_dissoc)

fig2, ax2 = plt.subplots() 
for col in data_dissoc.columns:
    if col != 'time_s': # We don't want to plot that stores our x-values
        ax2.plot(data_dissoc['time_s'], data_dissoc[col], ls='', marker='o', label=col)

ax2.legend()
ax2.set_xlabel('Time(s)')
ax2.set_ylabel('SPR signal')
plt.show()

In [ ]:
# Define the dissociation function, taking care to include the independent variable (t) first in the list
def dissociation(t, t_0, R_0, k_off):
    R_t = R_0*np.exp(-k_off*(t-t_0))
    return R_t

# We now have the function defined but nothing to do with it yet

In [ ]:
# Now, finally, let's run the  fit! Let's do this for just one condition, 125 nM, for the first run
from scipy.optimize import curve_fit # We need to import the curve_fit function

# to use curve_fit, we need to give it the function to fit to, x-values,
# y-values, and some optional parameters. In this case, I'm passing it initial
# guesses for the three fit paramenters (all as the variable p0) because the fit fails
# without these initial guesses
popt, pcov = curve_fit(dissociation, data_dissoc['time_s'], data_dissoc['125 nM'], p0=[600, 500, 0.001]) # This part does the fitting!

print (popt) # Show fit parameters
k_off_fit=popt[2] # The fit parameters in popt appear in the order they are passed to the function
print('The fit value for k_off is: ', k_off_fit, 's^(-1)')

y_fit = dissociation(data_dissoc['time_s'], popt[0], popt[1], popt[2]) # Calculate the fit curve for dissociation

fig3, ax3 = plt.subplots() 
ax3.plot(data_dissoc['time_s'], data_dissoc['125 nM'], ls='', marker='o', label='125 nM')
ax3.plot(data_dissoc['time_s'], y_fit, ls='-', marker='', label= '125 nM fit')

ax3.legend()
ax3.set_xlabel('Time(s)')
ax3.set_ylabel('SPR signal')
plt.show()


Great, now let's deal with the association part of the curve. The association part can be modeled as

$R(t) = R_{max}[1-e^{-(k_{on}C+k_{off})(t-t_0)}]$

Where:

$R_{max}$ is the maximal response,

$C$ is the concentration of the analyte, and

$k_{on}$ is the association rate constant, sometimes written as $k_1$


In [ ]:
# Define the association function:

def association(t, t_0, R_max, k_on, C=100, k_off=0.001):
    R_t = R_max * (1 - np.exp((-k_on * C + k_off) * (t - t_0)))
    return R_t




In [ ]:
from functools import partial # Importing a handy tool that will allow us to keep one fit parameter fixed

data_assoc = data[data['time_s'] < 600]

# Define parameters that should be fixed for this fit:
C_fixed = 1.25e-7 # 125 nM, converted to molar concentration
k_off_fixed = popt[2] # Result from our earlier fit of the dissociation rate

association_partial = partial(association, k_off=k_off_fixed, C=C_fixed) # define a new function with fixed C and k_off value

#Fit remaining free parameters:

popt2, pcov2 = curve_fit(association_partial, data_assoc['time_s'], data_assoc['125 nM'], p0=[0, 1000, 1e5]) # This part does the fitting!

print (popt2) # Show fit parameters
k_on_fit=popt2[2] # The fit parameters in popt appear in the order they are passed to the function
print('The fit value for k_on is:', k_on_fit, 'M^(-1) s^(-1)')

# Calculate the fit curve for dissociation
y_fit_assoc = association_partial(data_assoc['time_s'], popt2[0], popt2[1], popt2[2]) 


fig4, ax4 = plt.subplots() 
ax4.plot(data_assoc['time_s'], data_assoc['125 nM'], ls='', marker='o', label='125 nM')
ax4.plot(data_assoc['time_s'], y_fit_assoc, ls='-', marker='', label= '125 nM fit')

ax4.legend()
ax4.set_xlabel('Time(s)')
ax4.set_ylabel('SPR signal')
plt.show()


In [ ]:
# Finally, let's combine our k_on and k_off values to estimate the dissociation constant K_D.
# This is easy since K_D is just the ration of k_off to k_on:

K_D= k_off_fit / k_on_fit

print("The estimated K_D is:",K_D,"M")